# PricePilot-AI — Combined Exploratory Data Analysis

## Objective
This notebook analyzes **two complementary e-commerce datasets** for the PricePilot-AI dynamic pricing project.

- **Dataset 1:** Dynamic pricing / purchase-probability dataset
- **Dataset 2:** UCI Online Retail transaction dataset

The datasets are **not merged row-by-row** because they have different schemas and no common transaction key. Instead, each dataset is analyzed independently and their findings are connected at the business/feature level.

### Project flow
`EDA → data quality → patterns → feature understanding → ML-ready insights → dynamic pricing / revenue optimization`


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = "../data"  # change this only if your folder structure is different

dynamic_path = os.path.join(DATA_DIR, "ecommerce_dynamic_pricing_dataset.csv")
retail_path = os.path.join(DATA_DIR, "Online_Retail.xlsx")

df_dynamic = pd.read_csv(dynamic_path)
df_retail = pd.read_excel(retail_path)

print("Dynamic Pricing Dataset:", df_dynamic.shape)
print("Online Retail Dataset:", df_retail.shape)


## Part A — Dataset 1: Dynamic Pricing Dataset

In [ ]:
display(df_dynamic.head())
print("\nShape:", df_dynamic.shape)
print("\nData types:")
display(df_dynamic.dtypes.to_frame("dtype"))


In [ ]:
missing_dynamic = df_dynamic.isna().sum().sort_values(ascending=False)
display(missing_dynamic.to_frame("missing_values"))

print("Duplicate rows:", df_dynamic.duplicated().sum())
display(df_dynamic.describe(include="all").T)


### Categorical and numerical distributions

In [ ]:
for col in df_dynamic.select_dtypes(include="object").columns:
    print(f"\n{col} — {df_dynamic[col].nunique()} unique values")
    display(df_dynamic[col].value_counts().head(10).to_frame("count"))

numeric_dynamic = df_dynamic.select_dtypes(include=np.number).columns
display(df_dynamic[numeric_dynamic].describe().T)


In [ ]:
for col in ["Price", "Discount", "Customer_Age", "Review_Rating"]:
    if col in df_dynamic:
        plt.figure(figsize=(8,4))
        sns.histplot(df_dynamic[col], kde=True)
        plt.title(f"Distribution of {col}")
        plt.tight_layout()
        plt.show()


### Purchase probability relationships

In [ ]:
for col in ["Price", "Discount", "Customer_Age", "Review_Rating"]:
    if col in df_dynamic:
        plt.figure(figsize=(8,4))
        sns.scatterplot(data=df_dynamic, x=col, y="Purchase Probability", alpha=0.5)
        plt.title(f"{col} vs Purchase Probability")
        plt.tight_layout()
        plt.show()


In [ ]:
category_summary = df_dynamic.groupby("Product_Category").agg(
    Transactions=("Transaction_ID","count"),
    Avg_Price=("Price","mean"),
    Avg_Discount=("Discount","mean"),
    Avg_Rating=("Review_Rating","mean"),
    Purchase_Rate=("Purchase Probability","mean")
).sort_values("Purchase_Rate", ascending=False)

category_summary["Purchase_Rate_%"] = category_summary["Purchase_Rate"] * 100
display(category_summary.round(2))

plt.figure(figsize=(10,5))
sns.barplot(data=category_summary.reset_index(), x="Product_Category", y="Purchase_Rate_%")
plt.title("Purchase Rate by Product Category")
plt.ylabel("Purchase Rate (%)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
corr_cols = df_dynamic.select_dtypes(include=np.number).columns
plt.figure(figsize=(10,7))
sns.heatmap(df_dynamic[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix — Dynamic Pricing Dataset")
plt.tight_layout()
plt.show()


## Part B — Dataset 2: UCI Online Retail

In [ ]:
display(df_retail.head())
print("\nShape:", df_retail.shape)
print("\nData types:")
display(df_retail.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df_retail.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

print("\nDuplicate rows:", df_retail.duplicated().sum())


### Transaction cleaning indicators

In [ ]:
df_retail["InvoiceNo"] = df_retail["InvoiceNo"].astype(str)
df_retail["InvoiceDate"] = pd.to_datetime(df_retail["InvoiceDate"], errors="coerce")

df_retail["Is_Cancelled"] = df_retail["InvoiceNo"].str.startswith("C")
df_retail["Revenue"] = df_retail["Quantity"] * df_retail["UnitPrice"]

print("Cancelled invoice rows:", int(df_retail["Is_Cancelled"].sum()))
print("Rows with non-positive quantity:", int((df_retail["Quantity"] <= 0).sum()))
print("Rows with non-positive unit price:", int((df_retail["UnitPrice"] <= 0).sum()))

display(df_retail[["Quantity","UnitPrice","Revenue"]].describe().T)


### Revenue and demand analysis

In [ ]:
valid_retail = df_retail[
    (~df_retail["Is_Cancelled"]) &
    (df_retail["Quantity"] > 0) &
    (df_retail["UnitPrice"] > 0)
].copy()

print("Valid positive-sales rows:", len(valid_retail))
print("Unique products:", valid_retail["StockCode"].nunique())
print("Unique customers:", valid_retail["CustomerID"].nunique())
print("Countries:", valid_retail["Country"].nunique())

product_summary = valid_retail.groupby(["StockCode","Description"]).agg(
    Units_Sold=("Quantity","sum"),
    Revenue=("Revenue","sum"),
    Avg_UnitPrice=("UnitPrice","mean"),
    Transactions=("InvoiceNo","nunique")
).sort_values("Revenue", ascending=False)

display(product_summary.head(15).round(2))


In [ ]:
top_products = valid_retail.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=top_products.values, y=top_products.index)
plt.title("Top 10 Products by Units Sold")
plt.xlabel("Units Sold")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

country_sales = valid_retail.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=country_sales.values, y=country_sales.index)
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


### Time-based sales analysis

In [ ]:
valid_retail["Month"] = valid_retail["InvoiceDate"].dt.to_period("M").astype(str)
valid_retail["Hour"] = valid_retail["InvoiceDate"].dt.hour

monthly_sales = valid_retail.groupby("Month").agg(
    Revenue=("Revenue","sum"),
    Units=("Quantity","sum"),
    Transactions=("InvoiceNo","nunique")
).reset_index()

display(monthly_sales)

plt.figure(figsize=(12,5))
sns.lineplot(data=monthly_sales, x="Month", y="Revenue", marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

hourly_units = valid_retail.groupby("Hour")["Quantity"].sum().reset_index()

plt.figure(figsize=(10,5))
sns.lineplot(data=hourly_units, x="Hour", y="Quantity", marker="o")
plt.title("Units Sold by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()


### Customer-level analysis

In [ ]:
customer_summary = valid_retail.groupby("CustomerID").agg(
    Orders=("InvoiceNo","nunique"),
    Units=("Quantity","sum"),
    Revenue=("Revenue","sum"),
    Avg_UnitPrice=("UnitPrice","mean")
).sort_values("Revenue", ascending=False)

display(customer_summary.head(10).round(2))

plt.figure(figsize=(8,4))
sns.histplot(customer_summary["Revenue"], bins=40, kde=True)
plt.title("Distribution of Customer Revenue")
plt.xlabel("Customer Revenue")
plt.tight_layout()
plt.show()


## Part C — Outlier Analysis

In [ ]:
def iqr_report(df, columns):
    rows = []
    for col in columns:
        s = df[col].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
        n = int(((s < lower) | (s > upper)).sum())
        rows.append([col, n, round(n/len(s)*100,2), lower, upper])
    return pd.DataFrame(rows, columns=["Feature","Outlier_Count","Outlier_%","Lower_Bound","Upper_Bound"])

print("Dynamic pricing dataset:")
display(iqr_report(df_dynamic, ["Price","Discount","Customer_Age","Review_Rating"]).round(2))

print("Online Retail valid sales:")
display(iqr_report(valid_retail, ["Quantity","UnitPrice","Revenue"]).round(2))


## Part D — Combined Business Insights

### What Dataset 1 contributes
- Direct pricing variables such as **Price** and **Discount**.
- Customer attributes and **Purchase Probability**.
- Useful for learning the relationship between price/discount and predicted customer response.

### What Dataset 2 contributes
- Large-scale historical transaction behaviour.
- **Quantity, UnitPrice and Revenue** derived from actual purchases.
- Product, customer, country and time-based demand patterns.
- Useful for understanding sales volume, revenue and historical demand.

### Why we keep them as separate analyses
The two datasets do not share a reliable common key or identical schema, so directly concatenating/merging their rows would be misleading. Their value is complementary: Dataset 1 provides a pricing-response perspective, while Dataset 2 provides large-scale transaction and demand evidence.

### ML / PricePilot-AI direction
The EDA suggests a future pipeline where:
1. Transaction data is cleaned and aggregated into product/customer/time features.
2. Pricing-response variables are used to model purchase probability or demand.
3. Candidate prices can be evaluated using predicted demand/purchase response.
4. Revenue can be estimated as `candidate_price × predicted_quantity` (or an appropriate business objective).
5. The frontend can display the recommended price and supporting insights.

**Important:** EDA findings are observations, not proof of causality. Correlation between price and purchase behaviour does not by itself prove that changing price will cause the observed change.


## Final Takeaway for Milestone 1

> **EDA helped us understand both customer response to pricing and real historical transaction behaviour. Dataset 1 is useful for price-response/purchase-probability analysis, while Dataset 2 provides large-scale demand, product, customer, time and revenue patterns. Together, these insights provide the foundation for the next stage: feature engineering and machine-learning-based price/revenue optimization.**
